# 第Ⅱ部 8章 Gen AI Evaluation ServiceによるLLMとAIエージェントの品質評価

本ノートブックは、第Ⅱ部 8章のハンズオン用サンプルコードです。
Colab Enterprise 上で、上のセルから順に実行してください。

実行後は課金を避けるため、末尾の「クリーンアップ」セルを必ず実行してください。

## 8.6 環境構築

ライブラリのインストールと認証を行います。`pip install` の完了後は、依存関係を反映するためにランタイムを再起動してください。

In [ ]:
!pip install -q "google-cloud-aiplatform[adk,agent_engines,evaluation]==1.136.0" gcsfs

In [ ]:
import os

import vertexai
from vertexai import Client, types

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
# Gen AI Evaluation Service が利用可能なリージョンを明示します。
# Colab Enterprise では環境変数 GOOGLE_CLOUD_REGION にランタイムのリージョンが
# 自動設定されるため、そのまま参照すると非対応リージョン（例: asia-northeast1）で
# 「Unsupported region for Vertex Evaluation Service」エラーになることがあります。
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)
client = Client(project=PROJECT_ID, location=LOCATION)

## 8.7 評価対象データセットの準備

In [ ]:
import pandas as pd

eval_df = pd.DataFrame(
    {
        "prompt": [
            "短期記憶は脳のどの部分に依存していますか？",
            "Pythonでリストの重複を除去する最もシンプルな方法は何ですか？",
            "光合成のプロセスを小学生にもわかるように説明してください。",
        ],
        "reference": [
            "短期記憶は前頭葉（特に背外側前頭前野）と頭頂葉に依存しています。",
            "set() を使うのが最もシンプルです。例: list(set(my_list))",
            "光合成は植物が太陽の光と水と二酸化炭素を使って、栄養（ブドウ糖）と酸素を作るしくみです。",
        ],
    }
)
eval_df.head()

## 8.8 単一出力に対しての評価

In [ ]:
eval_dataset = client.evals.run_inference(
    model="gemini-2.5-flash",
    src=eval_df,
)

eval_dataset.show()

In [ ]:
eval_result = client.evals.evaluate(
    dataset=eval_dataset,
    metrics=[types.RubricMetric.TEXT_QUALITY],
)

eval_result.show()

In [ ]:
multi_metrics = [
    # ルーブリックベース
    types.RubricMetric.TEXT_QUALITY,
    types.RubricMetric.INSTRUCTION_FOLLOWING,
    types.RubricMetric.COHERENCE,
    # 計算ベース
    types.Metric(name="rouge_1"),
    types.Metric(name="bleu"),
]

multi_result = client.evals.evaluate(
    dataset=eval_dataset,
    metrics=multi_metrics,
)

multi_result.show()

In [ ]:
prompts_df = pd.DataFrame(
    {
        "prompt": [
            "一般相対性理論における時空の歪みがGPSの精度に与える影響を、具体的な補正値を含めて説明してください。",
            "日本語の敬語体系（尊敬語、謙譲語、丁寧語）がなぜ3分類では不十分とされ、5分類に改められたのか、その背景と具体例を説明してください。",
        ],
    }
)

result_a = client.evals.run_inference(
    model="gemini-2.5-flash", src=prompts_df
)
result_b = client.evals.run_inference(
    model="gemini-2.5-pro", src=prompts_df
)

comparison_result = client.evals.evaluate(
    dataset=[result_a, result_b],
    metrics=[
        types.RubricMetric.GENERAL_QUALITY,
        types.RubricMetric.TEXT_QUALITY,
    ],
)

comparison_result.show()

In [ ]:
GCS_DEST = f"gs://{PROJECT_ID}-genai-eval/single-generation-eval/batch/"

# 推論結果をGCSに保存
eval_dataset = client.evals.run_inference(
    model="gemini-2.5-flash",
    src=eval_df,
    config={"dest": GCS_DEST},
)

# バッチ評価を実行
batch_eval_job = client.evals.batch_evaluate(
    dataset=eval_dataset,
    metrics=[
        types.RubricMetric.FLUENCY,
        types.RubricMetric.COHERENCE,
        types.Metric(name="rouge_1"),
        types.Metric(name="bleu"),
    ],
    dest=GCS_DEST,
)

print(f"バッチ評価ジョブ: {batch_eval_job.name}")

In [ ]:
import json
import time

operation_name = batch_eval_job.name

while True:
    response = client._api_client.request("get", operation_name, {}, None)
    operation = json.loads(response.body) if response.body else {}

    if operation.get("done"):
        if "error" in operation:
            raise RuntimeError(
                f"ジョブ失敗 (code={operation['error'].get('code')}): "
                f"{operation['error'].get('message')}"
            )
        print("ジョブ完了!")
        completed_operation = operation
        break

    print(f"実行中... ({operation.get('metadata', {}).get('genericMetadata', {}).get('updateTime', '')})")
    time.sleep(30)

In [ ]:
# バッチ評価の結果をCloud Storageから読み込んで表示する。
# サンプルコードの view_batch_eval_results.py と同じ内容。

import json

import gcsfs
import pandas as pd
from IPython.display import display


def load_eval_results(gcs_dest: str, project_id: str):
    """GCS から最新の評価結果ファイルを読み込む。"""
    fs = gcsfs.GCSFileSystem(project=project_id)
    gcs_path = gcs_dest.replace("gs://", "")
    all_files = fs.find(gcs_path)
    agg_file = sorted(f for f in all_files if f.endswith("/aggregation_results.jsonl"))[-1]
    eval_file = sorted(f for f in all_files if f.endswith("/evaluation_results.jsonl"))[-1]
    return (
        pd.read_json(f"gs://{agg_file}", lines=True),
        pd.read_json(f"gs://{eval_file}", lines=True),
    )


def parse_aggregation(agg_raw: pd.DataFrame) -> pd.DataFrame:
    """集計結果 JSONL をメトリクス × 統計量のテーブルに整形する。"""
    melted = agg_raw.melt(
        id_vars=["aggregationMetric"], var_name="metric", value_name="result"
    )
    melted = melted[melted["result"].apply(lambda x: isinstance(x, dict))]
    melted["score"] = melted["result"].apply(lambda x: x["score"])
    summary = melted.pivot_table(
        index="metric", columns="aggregationMetric", values="score", aggfunc="first"
    )
    summary = summary.rename(
        columns={"AVERAGE": "average", "STANDARD_DEVIATION": "std_dev"}
    )
    summary.columns.name = None
    return summary


def parse_evaluation(eval_raw: pd.DataFrame) -> pd.DataFrame:
    """個別評価結果 JSONL をプロンプト × スコアのテーブルに整形する。"""
    instances = pd.json_normalize(
        eval_raw["jsonInstance"].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )
    )[["prompt", "response"]]
    scores = eval_raw["evaluationResults"].apply(
        lambda rs: {
            k: v["score"]
            for r in rs for k, v in r.items()
            if isinstance(v, dict) and "score" in v
        }
    )
    return pd.concat([instances, pd.DataFrame(scores.tolist())], axis=1)


agg_raw, eval_raw = load_eval_results(GCS_DEST, PROJECT_ID)  # noqa: F821

print("=== 集計結果 ===")
display(parse_aggregation(agg_raw))

print("\n=== 個別評価結果 ===")
display(parse_evaluation(eval_raw))

## 8.9 エージェントの評価

In [ ]:
def search_restaurant(keyword: str) -> dict:
    """キーワードからレストランを検索するサンプル。"""
    keywords = ["イタリアン", "フレンチ", "寿司", "和食", "個室", "記念日", "接待", "渋谷"]

    if any(kw in keyword for kw in keywords):
        return {"restaurants": [{"name": "トラットリア・ベッラ渋谷", "id": "R-TRATTORIA-SHIBUYA", "cuisine": "イタリアン", "area": "渋谷", "rating": 4.3}, {"name": "鮨なかむら恵比寿", "id": "R-SUSHI-EBISU", "cuisine": "寿司", "area": "恵比寿", "rating": 4.7}, {"name": "ビストロ・ルミエール表参道", "id": "R-BISTRO-OMOTESANDO", "cuisine": "フレンチ", "area": "表参道", "rating": 4.5}]}
    return {"restaurants": []}


def get_course_plan(restaurant_id: str, budget_per_person: int) -> dict:
    """予算に応じたコースを返すサンプル。"""
    if budget_per_person < 8000:
        return {"course": {"id": "C-TRATTORIA-CASUAL-001", "title": "カジュアルディナーコース", "price_per_person": 5500, "duration_minutes": 90, "includes_drink": False, "menu": ["本日の前菜3種盛り合わせ", "自家製フォカッチャ", "渡り蟹のトマトクリームパスタ", "本日のドルチェ"]}}
    return {"course": {"id": "C-TRATTORIA-PREMIUM-001", "title": "シェフおまかせプレミアムコース", "price_per_person": 12000, "duration_minutes": 120, "includes_drink": True, "menu": ["季節の冷前菜 〜鮮魚のカルパッチョ〜", "自家製ブッラータチーズと完熟トマトのカプレーゼ", "黒トリュフのタリアテッレ", "A5和牛ランプのタリアータルッコラ添え", "ティラミスと季節のフルーツ", "食後のエスプレッソ"]}}


def book_restaurant(
    restaurant_id: str,
    course_id: str,
    date: str,
    time: str,
    num_people: int,
) -> dict:
    """予約を作成するサンプル。"""
    reservation_id = (
        f"RV-{course_id}-{date.replace('-', '')}"
        f"-T{time.replace(':', '')}-N{num_people}"
    )
    return {"reservation": {"reservation_id": reservation_id, "restaurant_id": restaurant_id, "course_id": course_id, "date": date, "time": time, "num_people": num_people, "status": "CONFIRMED", "message": "ご予約が確定しました。当日のご来店をお待ちしております。"}}

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.planners import BuiltInPlanner
from google.genai import types as genai_types

root_agent = LlmAgent(
    model="gemini-2.5-flash",
    name="restaurant_booking_agent",
    instruction=(
        "あなたはレストラン予約エージェントです。"
        "下記のツールを使用してお客様からの要望に応えましょう。"
        "- お客様の希望（料理ジャンル、雰囲気、エリアなど）"
        "からレストランを検索するにはsearch_restaurantを必ず使用してください。"
        "- コースやプランを検索するにはget_course_planを必ず使用してください。"
        "- 予約を確定するにはbook_restaurantを必ず使用してください。"
    ),
    tools=[search_restaurant, get_course_plan, book_restaurant],
    planner=BuiltInPlanner(
        thinking_config=genai_types.ThinkingConfig(
            include_thoughts=False,
            thinking_budget=5000,
        )
    ),
)

In [ ]:
from google.genai import types as genai_types
from vertexai import agent_engines

# Agent評価ではv1beta1 APIを使用
client = Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=genai_types.HttpOptions(api_version="v1beta1"),
)

BUCKET_NAME = f"{PROJECT_ID}-genai-eval"
GCS_DEST = f"gs://{BUCKET_NAME}/adk-eval-sample"

app = agent_engines.AdkApp(agent=root_agent)

remote_app = client.agent_engines.create(
    agent=app,
    config={
        "staging_bucket": f"gs://{BUCKET_NAME}",
        "requirements": ["google-cloud-aiplatform[adk,agent_engines]"],
        "env_vars": {
            "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
            "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
        },
        "display_name": "restaurant_booking_agent",
        "resource_limits": {"cpu": "4", "memory": "8Gi"},
        "min_instances": 0,
        "max_instances": 1,
        "container_concurrency": 9,
    },
)

agent_engine_resource_name = remote_app.api_resource.name
print(f"デプロイ完了: {agent_engine_resource_name}")

In [ ]:
from datetime import datetime, timedelta

import pandas as pd
from vertexai import types

session_inputs = types.evals.SessionInput(
    user_id="test_user_001",
    state={},
)

reservation_date = datetime.now().strftime("%Y-%m-%d")

agent_prompts = [
    "記念日に使えるおしゃれなイタリアンのお店を探しています。",
    "渋谷のトラットリア・ベッラで、一人あたり5000円くらいのコースはありますか？",
    "トラットリア・ベッラ渋谷で予算一人15000円のコースを見せてください。",
    f"トラットリア・ベッラ渋谷のプレミアムコースを4名で予約したいです。{reservation_date} の19時でお願いします。",
]

agent_dataset = pd.DataFrame(
    {
        "prompt": agent_prompts,
        "session_inputs": [session_inputs] * len(agent_prompts),
    }
)
agent_dataset.head()

In [ ]:
# 推論の実行
agent_dataset_with_inference = client.evals.run_inference(
    agent=agent_engine_resource_name,
    src=agent_dataset,
)

In [ ]:
# Agent情報の取得
agent_info = types.evals.AgentInfo.load_from_agent(
    root_agent, agent_engine_resource_name
)
agent_info

In [ ]:
evaluation_run = client.evals.create_evaluation_run(
    # run_inference() で取得した推論済みデータセット
    dataset=agent_dataset_with_inference,
    # AgentInfoオブジェクト。評価基準の動的生成に使用される
    agent_info=agent_info,
    # 使用する評価メトリクスのリスト
    metrics=[
        # Adaptive: 最終応答の品質
        types.RubricMetric.FINAL_RESPONSE_QUALITY,
        # Adaptive: ツール選択、パラメーターの適切さ
        types.RubricMetric.TOOL_USE_QUALITY,
        # Static: ツール実行記録に対する事実性
        types.RubricMetric.HALLUCINATION,
        # 安全性
        types.RubricMetric.SAFETY,
        # Adaptive: カスタムガイドライン準拠
        types.RubricMetric.GENERAL_QUALITY(
            metric_spec_parameters={
                "guidelines": (
                    "常に丁寧かつ専門的な対応を心がけ、"
                    "レストランのコース提案やご予約手続きに専念し、"
                    "飲食以外の専門的な助言は行わない"
                )
            }
        ),
    ],
    # 評価結果の保存先Cloud Storageパス
    dest=GCS_DEST,
    # 評価ランの表示名
    display_name="restaurant_booking_agent_eval",
)

print(f"評価ラン作成: {evaluation_run.name}")
print(f"状態: {evaluation_run.state}")

In [ ]:
import time

POLL_INTERVAL = 10  # 秒
MAX_WAIT = 30 * 60  # 最大30分
ENDED_STATES = {"SUCCEEDED", "FAILED", "CANCELLED"}
elapsed = 0

while elapsed < MAX_WAIT:
    evaluation_run = client.evals.get_evaluation_run(name=evaluation_run.name)
    state = evaluation_run.state
    print(f"状態: {state}")

    if state in ENDED_STATES:
        if state == "SUCCEEDED":
            print("評価が正常に完了しました。")
        else:
            print(f"評価が異常終了しました: {evaluation_run.error}")
        break

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    print(f"警告: {MAX_WAIT // 60}分経過してもジョブが完了しませんでした。")

In [ ]:
evaluation_run = client.evals.get_evaluation_run(
    name=evaluation_run.name,
    include_evaluation_items=True,
)

evaluation_run.show()

### クリーンアップ

課金対象として残らないよう、Agent EngineリソースとCloud Storage上の評価結果を削除します。

In [ ]:
# 1. Agent Engineリソースの削除
client.agent_engines.delete(name=agent_engine_resource_name, force=True)

# 2. Cloud Storageのエージェント評価結果を削除
!gcloud storage rm -r {GCS_DEST}

# 3. Cloud Storageのバッチ評価結果を削除
!gcloud storage rm -r gs://{BUCKET_NAME}/single-generation-eval/